# RF Intrusion Detection — v3/v4 CI/CD Promotion and Batch Transform Pipeline

This notebook is the final CNN v3/v4 artifact-promotion workflow. It promotes an already-uploaded CNN v1–v4 `.keras` artifact; it does **not** retrain models and it does **not** use Feature Store. It follows the reference lab's pipeline pattern, adapted for the RF image workflow and SageMaker SDK v3. The earlier CNN v2 baseline Batch Transform demonstration is in Notebook 02.

**Safety:** Sections 1–8 are read-only or construct an in-memory pipeline definition. Section 9 contains the only AWS-mutating cells and is intentionally commented out. Review them before running.

## 1. Pipeline behavior and expected demonstrations

`Validate inputs → evaluate/package uploaded Keras model → performance gate → pass: prepare JSON payloads, register, create model, Batch Transform; fail: FailStep`.

Use v4 for the successful run. Its validated local scores are accuracy `0.8875` and macro F1 `0.8865`.  
Use v3 for a natural failed gate (accuracy `0.6204`, macro F1 `0.5931`). The failing branch cannot register a model package, create a SageMaker Model, or start Batch Transform.

### 1.1 Training scope and model provenance

CNN training is performed offline because image-model experimentation requires local iterative development and produces uploaded `.keras` candidate artifacts. This SageMaker pipeline implements controlled CI/CD promotion rather than continuous CNN training: it validates the curated input layout, evaluates each candidate against the held-out test split, enforces accuracy and macro-F1 thresholds, registers only approved candidates, and performs Batch Transform only for passing models.

CNN v3 is the intentional failed-gate demonstration. CNN v4 is the improved candidate and successful-promotion demonstration.

## 2. SDK v3 setup and local source checks

This notebook deliberately uses SDK v3 module paths. This cell does not call AWS.

In [1]:
import base64
import json
from pathlib import Path

import boto3
import sagemaker
from sagemaker.core import image_uris
from sagemaker.core.model_metrics import MetricsSource, ModelMetrics
from sagemaker.core.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.core.transformer import Transformer
from sagemaker.core.workflow import (
    ConditionGreaterThanOrEqualTo,
    Join,
    JsonGet,
    ParameterFloat,
    ParameterString,
    PropertyFile,
)
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.serve import ModelBuilder
from sagemaker.mlops.workflow.condition_step import ConditionStep
from sagemaker.mlops.workflow.fail_step import FailStep
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TransformStep

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / ".config" / "project.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CONFIG_PATH}. Copy .config/project.example.json to .config/project.json and fill in your AWS values."
    )
with CONFIG_PATH.open(encoding="utf-8") as config_file:
    PROJECT_CONFIG = json.load(config_file)
required_config_keys = {
    "aws_region", "s3_bucket", "role_arn", "data_lake_prefix", "model_prefix",
    "pipeline_prefix", "pipeline_name", "model_package_group", "batch_output_prefix",
}
missing_config_keys = required_config_keys - PROJECT_CONFIG.keys()
if missing_config_keys:
    raise KeyError(f"Missing project configuration keys: {sorted(missing_config_keys)}")
if any("YOUR_" in str(PROJECT_CONFIG[key]) for key in ("s3_bucket", "role_arn")):
    raise ValueError("Replace placeholder values in .config/project.json before running this notebook.")

print("SDK v3 imports and project configuration loaded.")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
SDK v3 imports loaded.


In [2]:
CODE_FILENAMES = (
    "validate_rf_intrusion_inputs.py",
    "evaluate_and_package_rf_intrusion.py",
    "prepare_batch_json_payloads.py",
)

EMBEDDED_PIPELINE_SOURCES = {
    "validate_rf_intrusion_inputs.py": r'''
import argparse
import json
from pathlib import Path

CLASSES = [
    "Benign", "Broadband_Jam", "Human",
    "Machine", "Narrowband_Jam", "Wildlife",
]

def validate_split(name, root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"{name} directory does not exist: {root}")

    found = sorted(path.name for path in root.iterdir() if path.is_dir())
    if found != CLASSES:
        raise ValueError(f"{name} classes must be {CLASSES}; found {found}")

    counts = {
        label: sum(1 for _ in (root / label).glob("*.png"))
        for label in CLASSES
    }
    if any(count == 0 for count in counts.values()):
        raise ValueError(f"{name} contains an empty class: {counts}")
    return counts

parser = argparse.ArgumentParser()
parser.add_argument("--train-dir", default="/opt/ml/processing/train")
parser.add_argument("--validation-dir", default="/opt/ml/processing/validation")
parser.add_argument("--test-dir", default="/opt/ml/processing/test")
parser.add_argument(
    "--output-dir",
    default="/opt/ml/processing/validation-report",
)
args = parser.parse_args()

manifest = {
    "class_names": CLASSES,
    "image_counts": {
        "train": validate_split("train", args.train_dir),
        "validation": validate_split("validation", args.validation_dir),
        "test": validate_split("test", args.test_dir),
    },
}

output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "input_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)
print(json.dumps(manifest, indent=2))
''',

    "evaluate_and_package_rf_intrusion.py": r'''
import argparse
import json
import tarfile
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

CLASSES = [
    "Benign", "Broadband_Jam", "Human",
    "Machine", "Narrowband_Jam", "Wildlife",
]

parser = argparse.ArgumentParser()
parser.add_argument("--model-version", required=True)
parser.add_argument(
    "--model-input-dir",
    default="/opt/ml/processing/model-input",
)
parser.add_argument("--test-dir", default="/opt/ml/processing/test")
parser.add_argument(
    "--evaluation-dir",
    default="/opt/ml/processing/evaluation",
)
parser.add_argument(
    "--package-dir",
    default="/opt/ml/processing/model-package",
)
args = parser.parse_args()

keras_files = list(Path(args.model_input_dir).rglob("*.keras"))
if len(keras_files) != 1:
    raise ValueError(
        f"Expected exactly one .keras artifact; found {keras_files}"
    )

model = tf.keras.models.load_model(keras_files[0])
test_ds = tf.keras.utils.image_dataset_from_directory(
    args.test_dir,
    image_size=(128, 128),
    batch_size=32,
    label_mode="categorical",
    color_mode="rgb",
    shuffle=False,
)

if test_ds.class_names != CLASSES:
    raise ValueError(
        f"Expected class order {CLASSES}; got {test_ds.class_names}"
    )

y_true = np.concatenate(
    [np.argmax(labels.numpy(), axis=1) for _, labels in test_ds]
)
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

metrics = {
    "model_version": args.model_version,
    "class_names": CLASSES,
    "multiclass_classification_metrics": {
        "accuracy": {"value": float(accuracy_score(y_true, y_pred))},
        "macro_f1": {"value": float(macro_f1)},
        "macro_precision": {"value": float(precision)},
        "macro_recall": {"value": float(recall)},
    },
    "classification_report": classification_report(
        y_true,
        y_pred,
        labels=range(len(CLASSES)),
        target_names=CLASSES,
        output_dict=True,
        zero_division=0,
    ),
    "confusion_matrix": confusion_matrix(
        y_true, y_pred, labels=range(len(CLASSES))
    ).tolist(),
}

evaluation_dir = Path(args.evaluation_dir)
evaluation_dir.mkdir(parents=True, exist_ok=True)
(evaluation_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

package_dir = Path(args.package_dir)
package_dir.mkdir(parents=True, exist_ok=True)
serving_dir = package_dir / "1"

if hasattr(model, "export"):
    model.export(serving_dir)
else:
    tf.saved_model.save(model, str(serving_dir))

with tarfile.open(package_dir / "model.tar.gz", "w:gz") as archive:
    archive.add(serving_dir, arcname="1")

print(json.dumps(metrics["multiclass_classification_metrics"], indent=2))
''',

    "prepare_batch_json_payloads.py": r'''
import argparse
import json
from pathlib import Path

import numpy as np
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument(
    "--input-dir",
    default="/opt/ml/processing/production",
)
parser.add_argument(
    "--output-dir",
    default="/opt/ml/processing/batch-input",
)
args = parser.parse_args()

source_dir = Path(args.input_dir)
output_dir = Path(args.output_dir)
png_files = sorted(source_dir.rglob("*.png"))

if not png_files:
    raise FileNotFoundError(f"No PNG files found under {source_dir}")

for png_path in png_files:
    relative_json = png_path.relative_to(source_dir).with_suffix(".json")
    destination = output_dir / relative_json
    destination.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(png_path) as image:
        array = np.asarray(
            image.convert("RGB").resize((128, 128)),
            dtype=np.float32,
        )

    destination.write_text(json.dumps({"instances": [array.tolist()]}))

manifest = {
    "records": len(png_files),
    "input_shape": [128, 128, 3],
    "scale": "0-255",
}

# Do not save the manifest under batch-input: Batch Transform sends every
# object in that prefix to TensorFlow Serving, which requires `instances`.
print(json.dumps(manifest, indent=2))
''',
}

assert set(EMBEDDED_PIPELINE_SOURCES) == set(CODE_FILENAMES)
print("Embedded Processing code restored in memory only.")

Embedded Processing code restored in memory only.


## 3. Project configuration

All locations use the supplied account, Region, and slash-separated `rf-intrusion/datalake` prefix. `1/` inside the serving archive is the TensorFlow Serving version; it is not CNN v1.

In [3]:
REGION = PROJECT_CONFIG['aws_region']
BUCKET = PROJECT_CONFIG['s3_bucket']
ROLE_ARN = PROJECT_CONFIG['role_arn']
ACCOUNT_ID = ROLE_ARN.split(':')[4]
PIPELINE_NAME = PROJECT_CONFIG['pipeline_name']
MODEL_PACKAGE_GROUP = PROJECT_CONFIG['model_package_group']
DATA_LAKE_PREFIX = PROJECT_CONFIG['data_lake_prefix'].strip('/')
MODEL_PREFIX = PROJECT_CONFIG['model_prefix'].strip('/')
PIPELINE_PREFIX = PROJECT_CONFIG['pipeline_prefix'].strip('/')
CLASS_NAMES = ['Benign', 'Broadband_Jam', 'Human', 'Machine', 'Narrowband_Jam', 'Wildlife']

MODEL_URIS = {
    'v1': f's3://{BUCKET}/{MODEL_PREFIX}/v1/baseline_cnn_rf_intrusion_v1.keras',
    'v2': f's3://{BUCKET}/{MODEL_PREFIX}/v2/improved_cnn_rf_intrusion_v2.keras',
    'v3': f's3://{BUCKET}/{MODEL_PREFIX}/v3/rf_intrusion_cnn_v3.keras',
    'v4': f's3://{BUCKET}/{MODEL_PREFIX}/v4/rf_intrusion_cnn_v4.keras',
}
TRAIN_URI = f's3://{BUCKET}/{DATA_LAKE_PREFIX}/curated/assignment_split/train/'
VALIDATION_URI = f's3://{BUCKET}/{DATA_LAKE_PREFIX}/curated/assignment_split/validation/'
TEST_URI = f's3://{BUCKET}/{DATA_LAKE_PREFIX}/curated/assignment_split/test/'
PRODUCTION_URI = f's3://{BUCKET}/{DATA_LAKE_PREFIX}/curated/assignment_split/production/'
PIPELINE_OUTPUT_URI = f's3://{BUCKET}/{PIPELINE_PREFIX}/promotion/v4'
BATCH_OUTPUT_URI = f"s3://{BUCKET}/{PROJECT_CONFIG['batch_output_prefix'].strip('/')}/pipeline/v4/"
CODE_PREFIX = f'{PIPELINE_PREFIX}/promotion/notebook-code/v4'

CODE_URIS = {
    name: f"s3://{BUCKET}/{CODE_PREFIX}/{name}"
    for name in CODE_FILENAMES
}

assert list(CLASS_NAMES) == sorted(CLASS_NAMES), 'Class order must remain alphabetical.'
print('Configured:', PIPELINE_NAME, REGION, ROLE_ARN)

Configured: rf-intrusion-cnn-artifact-promotion-batch-transform us-east-1 arn:aws:iam::182280021178:role/LabRole


## 4. Read-only S3 path preflight

Run this before any pipeline update. It uses `HeadObject` and `ListObjectsV2` only; it creates no resources and writes nothing. It checks the selected v4 artifact by default and every required curated-data prefix.

In [4]:
# READ-ONLY AWS CALLS — HeadObject and ListObjectsV2 only.
from urllib.parse import urlparse

def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != 's3' or not parsed.netloc or not parsed.path.lstrip('/'):
        raise ValueError(f'Invalid S3 URI: {uri}')
    return parsed.netloc, parsed.path.lstrip('/')

s3_readonly = boto3.client('s3', region_name=REGION)
bucket_name, model_key = split_s3_uri(MODEL_URIS['v4'])

for version in ("v3", "v4"):
    bucket_name, model_key = split_s3_uri(MODEL_URIS[version])
    s3_readonly.head_object(Bucket=bucket_name, Key=model_key)
    print(f"OK {version}:", MODEL_URIS[version])

for prefix_uri in (TRAIN_URI, VALIDATION_URI, TEST_URI, PRODUCTION_URI):
    prefix_bucket, prefix = split_s3_uri(prefix_uri)
    if not s3_readonly.list_objects_v2(Bucket=prefix_bucket, Prefix=prefix, MaxKeys=1).get('Contents'):
        raise FileNotFoundError(f'No objects found under {prefix_uri}')
    print('OK:', prefix_uri)

OK v3: s3://sagemaker-us-east-1-182280021178/rf-intrusion/models/tensorflow/v3/rf_intrusion_cnn_v3.keras
OK v4: s3://sagemaker-us-east-1-182280021178/rf-intrusion/models/tensorflow/v4/rf_intrusion_cnn_v4.keras
OK: s3://sagemaker-us-east-1-182280021178/rf-intrusion/datalake/curated/assignment_split/train/
OK: s3://sagemaker-us-east-1-182280021178/rf-intrusion/datalake/curated/assignment_split/validation/
OK: s3://sagemaker-us-east-1-182280021178/rf-intrusion/datalake/curated/assignment_split/test/
OK: s3://sagemaker-us-east-1-182280021178/rf-intrusion/datalake/curated/assignment_split/production/


## 5. Pipeline parameters and container images

TensorFlow 2.19 is used for Keras 3 `model.export()` compatibility. The processing image performs validation, evaluation, serving-package creation, and production JSON payload preparation. The inference image serves the packaged `model.tar.gz` through TensorFlow Serving.

In [5]:
pipeline_session = PipelineSession(boto_session=boto3.Session(region_name=REGION))
tf_processing_image = image_uris.retrieve(framework='tensorflow', region=REGION, version='2.19.0', image_scope='training', instance_type='ml.m5.xlarge')
tf_inference_image = image_uris.retrieve(framework='tensorflow', region=REGION, version='2.19.0', image_scope='inference', instance_type='ml.m5.xlarge')

model_artifact_uri = ParameterString('ModelArtifactS3Uri', default_value=MODEL_URIS['v4'])
model_version = ParameterString('ModelVersion', default_value='v4')
model_approval_status = ParameterString('ModelApprovalStatus', default_value='PendingManualApproval')
macro_f1_threshold = ParameterFloat('MacroF1Threshold', default_value=0.86)
accuracy_threshold = ParameterFloat('AccuracyThreshold', default_value=0.87)
processing_instance_type = ParameterString('ProcessingInstanceType', default_value='ml.m5.xlarge')
transform_instance_type = ParameterString('TransformInstanceType', default_value='ml.m5.xlarge')
pipeline_output_uri = ParameterString('PipelineOutputS3Uri', default_value=PIPELINE_OUTPUT_URI)
batch_output_uri = ParameterString('BatchOutputS3Uri', default_value=BATCH_OUTPUT_URI)

[10/12/26 20:01:42] INFO     Defaulting to only available Python version: py312                   ]8;id=7622705;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=7622706;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

## 6. Validation and evaluation/package Processing Steps

The validator checks every required split has the exact six classes and at least one PNG per class. The evaluator loads exactly one selected Keras artifact, evaluates it against the unchanged test split, writes pipeline-readable `metrics.json`, and emits `model.tar.gz` whose top-level deployment directory is `1/`.

In [6]:
def processing_input(input_name, s3_uri, local_path):
    return ProcessingInput(
        input_name=input_name,
        s3_input={
            "s3_uri": s3_uri,
            "local_path": local_path,
            "s3_data_type": "S3Prefix",
            "s3_input_mode": "File",
        },
    )


def processing_output(output_name, s3_uri, local_path):
    return ProcessingOutput(
        output_name=output_name,
        s3_output={
            "s3_uri": s3_uri,
            "local_path": local_path,
            "s3_upload_mode": "EndOfJob",
        },
    )


validator = ScriptProcessor(
    image_uri=tf_processing_image,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    role=ROLE_ARN,
    sagemaker_session=pipeline_session,
)

validate_args = validator.run(
    inputs=[
        processing_input("train", TRAIN_URI, "/opt/ml/processing/train"),
        processing_input(
            "validation",
            VALIDATION_URI,
            "/opt/ml/processing/validation",
        ),
        processing_input("test", TEST_URI, "/opt/ml/processing/test"),
    ],
    outputs=[
        processing_output(
            "validation-report",
            Join(on="/", values=[pipeline_output_uri, "validation"]),
            "/opt/ml/processing/validation-report",
        )
    ],
    code=CODE_URIS["validate_rf_intrusion_inputs.py"],
)

step_validate = ProcessingStep(
    name="ValidateRFIntrusionInputs",
    step_args=validate_args,
)

packager = ScriptProcessor(
    image_uri=tf_processing_image,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    role=ROLE_ARN,
    sagemaker_session=pipeline_session,
)

package_args = packager.run(
    inputs=[
        processing_input(
            "model-input",
            model_artifact_uri,
            "/opt/ml/processing/model-input",
        ),
        processing_input("test", TEST_URI, "/opt/ml/processing/test"),
    ],
    outputs=[
        processing_output(
            "evaluation",
            Join(on="/", values=[pipeline_output_uri, "evaluation"]),
            "/opt/ml/processing/evaluation",
        ),
        processing_output(
            "model-package",
            Join(on="/", values=[pipeline_output_uri, "model-package"]),
            "/opt/ml/processing/model-package",
        ),
    ],
    code=CODE_URIS["evaluate_and_package_rf_intrusion.py"],
    arguments=["--model-version", model_version],
)

metrics_file = PropertyFile(
    name="RFIntrusionMetrics",
    output_name="evaluation",
    path="metrics.json",
)

step_evaluate_package = ProcessingStep(
    name="EvaluateAndPackageRFIntrusionCNN",
    step_args=package_args,
    property_files=[metrics_file],
    depends_on=[step_validate.name],
)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7622713;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7622714;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#322\322]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 7. Pass-only payload preparation, Registry, Model, and Batch Transform

Raw production PNGs are not valid TensorFlow Serving requests. The payload preparation step converts each one to a 128×128 RGB float JSON record in the `instances` format, retaining the 0–255 scale because the CNN includes its own rescaling layer. The registered Model Package version represents CNN v1–v4 through metadata; its archive always uses serving directory `1/`.

In [7]:
payload_processor = ScriptProcessor(
    image_uri=tf_processing_image,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    role=ROLE_ARN,
    sagemaker_session=pipeline_session,
)

payload_args = payload_processor.run(
    inputs=[
        processing_input(
            "production",
            PRODUCTION_URI,
            "/opt/ml/processing/production",
        )
    ],
    outputs=[
        processing_output(
            "batch-input",
            Join(on="/", values=[pipeline_output_uri, "batch-input"]),
            "/opt/ml/processing/batch-input",
        )
    ],
    code=CODE_URIS["prepare_batch_json_payloads.py"],
)

step_prepare_payloads = ProcessingStep(
    name="PrepareRFIntrusionBatchJson",
    step_args=payload_args,
)

packaged_model_uri = Join(
    on="/",
    values=[
        step_evaluate_package.properties.ProcessingOutputConfig.Outputs[
            "model-package"
        ].S3Output.S3Uri,
        "model.tar.gz",
    ],
)

model_builder = ModelBuilder(
    image_uri=tf_inference_image,
    s3_model_data_url=packaged_model_uri,
    role_arn=ROLE_ARN,
    sagemaker_session=pipeline_session,
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_evaluate_package.properties.ProcessingOutputConfig.Outputs[
                    "evaluation"
                ].S3Output.S3Uri,
                "metrics.json",
            ],
        ),
        content_type="application/json",
    )
)

register_args = model_builder.register(
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
    customer_metadata_properties={
        "cnn_version": model_version,
        "source_keras_s3_uri": model_artifact_uri,
    },
)

step_register = ModelStep(
    name="RegisterRFIntrusionCNN",
    step_args=register_args,
)

create_args = model_builder.build()

step_create_model = ModelStep(
    name="CreateRFIntrusionBatchModel",
    step_args=create_args,
)
step_create_model.add_depends_on([step_register])

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type=transform_instance_type,
    instance_count=1,
    output_path=batch_output_uri,
    sagemaker_session=pipeline_session,
)

transform_args = transformer.transform(
    data=step_prepare_payloads.properties.ProcessingOutputConfig.Outputs[
        "batch-input"
    ].S3Output.S3Uri,
    content_type="application/json",
    split_type="None",
)

step_transform = TransformStep(
    name="BatchTransformRFIntrusionCNN",
    step_args=transform_args,
)
step_transform.add_depends_on([step_prepare_payloads])

[10/12/26 20:01:43] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=7622721;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=7622722;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=7622728;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=7622729;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

                    DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=7622735;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=7622736;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1335\1335]8;;\
                             MLflow model input                                                                    

                    INFO     ✅ Model has been created: 'model-eb792234' using server None in ]8;id=7622743;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=7622744;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3357\3357]8;;\
                             SAGEMAKER_ENDPOINT mode                                                               

## 8. Performance gate, DAG definition, and static inspection

Both performance conditions must pass. The failure reason contains the selected version, both observed metrics, and both thresholds, so a v3 presentation has an unambiguous audit trail. Pipeline definition inspection is safe; do not run Section 9 until you approve the generated JSON and IAM access.

In [8]:
accuracy_value = JsonGet(step_name=step_evaluate_package.name, property_file=metrics_file, json_path='multiclass_classification_metrics.accuracy.value')
macro_f1_value = JsonGet(step_name=step_evaluate_package.name, property_file=metrics_file, json_path='multiclass_classification_metrics.macro_f1.value')
step_fail = FailStep(name='RFIntrusionPerformanceGateFailed', error_message=Join(on=' ', values=['RF Intrusion promotion failed.', 'ModelVersion=', model_version, 'accuracy=', accuracy_value, 'required_accuracy>=', accuracy_threshold, 'macro_f1=', macro_f1_value, 'required_macro_f1>=', macro_f1_threshold]))
step_gate = ConditionStep(name='RFIntrusionPerformanceGate', conditions=[ConditionGreaterThanOrEqualTo(left=macro_f1_value, right=macro_f1_threshold), ConditionGreaterThanOrEqualTo(left=accuracy_value, right=accuracy_threshold)], if_steps=[step_prepare_payloads, step_register, step_create_model, step_transform], else_steps=[step_fail])

pipeline = Pipeline(name=PIPELINE_NAME, parameters=[model_artifact_uri, model_version, model_approval_status, macro_f1_threshold, accuracy_threshold, processing_instance_type, transform_instance_type, pipeline_output_uri, batch_output_uri], steps=[step_validate, step_evaluate_package, step_gate], sagemaker_session=pipeline_session)
definition = json.loads(pipeline.definition())
print('Top-level DAG steps:', [step['Name'] for step in definition['Steps']])
print(json.dumps(definition, indent=2)[:5000])

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622751;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622752;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622757;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622758;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622763;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622764;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=7622771;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=7622772;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7622777;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622778;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=7622783;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622784;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

                    WARNING  Popping out 'TransformJobName' from the pipeline definition by        ]8;id=7622789;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622790;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Top-level DAG steps: ['ValidateRFIntrusionInputs', 'EvaluateAndPackageRFIntrusionCNN', 'RFIntrusionPerformanceGate']
{
  "Version": "2020-12-01",
  "Metadata": {},
  "Parameters": [
    {
      "Name": "ModelArtifactS3Uri",
      "Type": "String",
      "DefaultValue": "s3://sagemaker-us-east-1-182280021178/rf-intrusion/models/tensorflow/v4/rf_intrusion_cnn_v4.keras"
    },
    {
      "Name": "ModelVersion",
      "Type": "String",
      "DefaultValue": "v4"
    },
    {
      "Name": "ModelApprovalStatus",
      "Type": "String",
      "DefaultValue": "PendingManualApproval"
    },
    {
      "Name": "MacroF1Threshold",
      "Type": "Float",
      "DefaultValue": 0.86
    },
    {
      "Name": "AccuracyThreshold",
      "Type": "Float",
      "DefaultValue": 0.87
    },
    {
      "Name": "ProcessingInstanceType",
      "Type": "String",
      "DefaultValue": "ml.m5.xlarge"
    },
    {
      "Name": "TransformInstanceType",
      "Type": "String",
      "DefaultValue": "ml.m5.xl

In [9]:
dag = """
ValidateRFIntrusionInputs
          |
          v
EvaluateAndPackageRFIntrusionCNN
          |
          v
RFIntrusionPerformanceGate
     /                         \\
  pass                         fail
   |                            |
   v                            v
PrepareRFIntrusionBatchJson  RFIntrusionPerformanceGateFailed
   |
   v
RegisterRFIntrusionCNN
   |
   v
CreateRFIntrusionBatchModel
   |
   v
BatchTransformRFIntrusionCNN
"""

print(dag)


ValidateRFIntrusionInputs
          |
          v
EvaluateAndPackageRFIntrusionCNN
          |
          v
RFIntrusionPerformanceGate
     /                         \
  pass                         fail
   |                            |
   v                            v
PrepareRFIntrusionBatchJson  RFIntrusionPerformanceGateFailed
   |
   v
RegisterRFIntrusionCNN
   |
   v
CreateRFIntrusionBatchModel
   |
   v
BatchTransformRFIntrusionCNN



## 9. AWS-mutating deployment and demonstrations — run only after review

The following cells are disabled by default. They create/update the pipeline, create the Model Package Group only if absent, and start executions. They do not create an endpoint. Run the v4 success cell first, then v3. `v3` naturally fails the performance gate.

In [10]:
from botocore.exceptions import ClientError

# AWS-MUTATING: stage embedded Processing code to project S3.
s3_write = boto3.client("s3", region_name=REGION)

for filename, source in EMBEDDED_PIPELINE_SOURCES.items():
    s3_write.put_object(
        Bucket=BUCKET,
        Key=f"{CODE_PREFIX}/{filename}",
        Body=source.encode("utf-8"),
        ContentType="text/x-python",
    )

print("Staged code:")
for uri in CODE_URIS.values():
    print(" ", uri)

# AWS-MUTATING: create the Registry group only if absent.
sm = boto3.client("sagemaker", region_name=REGION)

try:
    sm.describe_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP
    )
    print("Reusing Model Package Group:", MODEL_PACKAGE_GROUP)

except ClientError as error:
    if error.response["Error"]["Code"] != "ValidationException":
        raise

    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription=(
            "RF Intrusion Detection CNN artifact promotion versions"
        ),
    )
    print("Created Model Package Group:", MODEL_PACKAGE_GROUP)

# AWS-MUTATING: create or update the Pipeline definition.
pipeline.upsert(role_arn=ROLE_ARN)

pipeline_description = sm.describe_pipeline(
    PipelineName=PIPELINE_NAME
)

print("Pipeline ARN:", pipeline_description["PipelineArn"])
print("Last modified:", pipeline_description["LastModifiedTime"])

Staged code:
  s3://sagemaker-us-east-1-182280021178/rf-intrusion/pipelines/promotion/notebook-code/v2/validate_rf_intrusion_inputs.py
  s3://sagemaker-us-east-1-182280021178/rf-intrusion/pipelines/promotion/notebook-code/v2/evaluate_and_package_rf_intrusion.py
  s3://sagemaker-us-east-1-182280021178/rf-intrusion/pipelines/promotion/notebook-code/v2/prepare_batch_json_payloads.py
Reusing Model Package Group: rf-intrusion-cnn-models


                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622795;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622796;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622801;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622802;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622807;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622808;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7622813;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622814;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

...

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=7622843;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622844;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=7622849;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622850;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=7622855;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622856;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

                    WARNING  Popping out 'TransformJobName' from the pipeline definition by        ]8;id=7622861;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=7622862;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Pipeline ARN: arn:aws:sagemaker:us-east-1:182280021178:pipeline/rf-intrusion-cnn-artifact-promotion-batch-transform
Last modified: 2026-10-12 20:01:44.593000+00:00


In [11]:
# AWS-MUTATING — run the v4 block first.
# This starts Processing Jobs and, if v4 passes the gate, creates a Model Package,
# SageMaker Model, prepared JSON payloads, and a Batch Transform job.

v4_start = sm.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineParameters=[
        {
            "Name": "ModelArtifactS3Uri",
            "Value": MODEL_URIS["v4"],
        },
        {
            "Name": "ModelVersion",
            "Value": "v4",
        },
        {
            "Name": "ModelApprovalStatus",
            "Value": "Approved",
        },
        {
            "Name": "MacroF1Threshold",
            "Value": "0.86",
        },
        {
            "Name": "AccuracyThreshold",
            "Value": "0.87",
        },
    ],
)

V4_EXECUTION_ARN = v4_start["PipelineExecutionArn"]
print("Started v4 execution:", V4_EXECUTION_ARN)

Started v4 execution: arn:aws:sagemaker:us-east-1:182280021178:pipeline/rf-intrusion-cnn-artifact-promotion-batch-transform/execution/iqgrcx3rplpg


## 10. Read-only deployment inspection

In [18]:
sm = boto3.client("sagemaker", region_name=REGION)
group = sm.describe_model_package_group(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP
)

print("Group ARN:", group["ModelPackageGroupArn"])
print("Description:", group["ModelPackageGroupDescription"])

packages = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageType="Versioned",
    SortBy="CreationTime",
    SortOrder="Ascending",
)

print("Registered package versions:", len(packages["ModelPackageSummaryList"]))

Group ARN: arn:aws:sagemaker:us-east-1:182280021178:model-package-group/rf-intrusion-cnn-models
Description: RF Intrusion Detection CNN artifact promotion versions
Registered package versions: 1


## 11. Read-only execution evidence

After an execution completes, use the following snippets to show the executed steps and, for v4, the generated metrics. For a failing v3 run, inspect `RFIntrusionPerformanceGateFailed` to show the explicit reason.

In [32]:
# READ-ONLY: set this to V4_EXECUTION_ARN or V3_EXECUTION_ARN.
EXECUTION_ARN = V4_EXECUTION_ARN

execution_details = sm.describe_pipeline_execution(
    PipelineExecutionArn=EXECUTION_ARN
)

print("Execution ARN:", EXECUTION_ARN)
print("Status:", execution_details["PipelineExecutionStatus"])
print("Started:", execution_details["CreationTime"])
print("Failure reason:", execution_details.get("FailureReason", "None"))

step_response = sm.list_pipeline_execution_steps(
    PipelineExecutionArn=EXECUTION_ARN
)

print("\nPipeline steps:")
for step in step_response["PipelineExecutionSteps"]:
    print(
        f"- {step['StepName']}: {step['StepStatus']}"
        f" | {step.get('FailureReason', '')}"
    )

Execution ARN: arn:aws:sagemaker:us-east-1:182280021178:pipeline/rf-intrusion-cnn-artifact-promotion-batch-transform/execution/iqgrcx3rplpg
Status: Succeeded
Started: 2026-10-12 20:01:44.969000+00:00
Failure reason: None

Pipeline steps:
- BatchTransformRFIntrusionCNN: Succeeded | 
- CreateRFIntrusionBatchModel: Succeeded | 
- RegisterRFIntrusionCNN: Succeeded | 
- PrepareRFIntrusionBatchJson: Succeeded | 
- RFIntrusionPerformanceGate: Succeeded | 
- EvaluateAndPackageRFIntrusionCNN: Succeeded | 
- ValidateRFIntrusionInputs: Succeeded | 


In [33]:
from urllib.parse import urlparse

metrics_uri = f"{PIPELINE_OUTPUT_URI}/evaluation/metrics.json"
parsed = urlparse(metrics_uri)

metrics = json.loads(
    s3_readonly.get_object(
        Bucket=parsed.netloc,
        Key=parsed.path.lstrip("/"),
    )["Body"].read()
)

print(json.dumps(
    metrics["multiclass_classification_metrics"],
    indent=2,
))

{
  "accuracy": {
    "value": 0.8870833333333333
  },
  "macro_f1": {
    "value": 0.8861113324281487
  },
  "macro_precision": {
    "value": 0.8877130120031055
  },
  "macro_recall": {
    "value": 0.8870833333333333
  }
}


## 12. Failure State - CNN v3

In [34]:
# AWS-MUTATING — controlled v3 performance-gate failure.
# This runs validation and test evaluation only. It does NOT register a model,
# create a SageMaker Model, prepare batch payloads, or run Batch Transform.

V3_PIPELINE_OUTPUT_URI = (
    f"s3://{BUCKET}/rf-intrusion/pipelines/promotion/v3-failure-demo"
)

v3_start = sm.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineParameters=[
        {
            "Name": "ModelArtifactS3Uri",
            "Value": MODEL_URIS["v3"],
        },
        {
            "Name": "ModelVersion",
            "Value": "v3",
        },
        {
            "Name": "ModelApprovalStatus",
            "Value": "PendingManualApproval",
        },
        {
            "Name": "MacroF1Threshold",
            "Value": "0.86",
        },
        {
            "Name": "AccuracyThreshold",
            "Value": "0.87",
        },
        {
            "Name": "PipelineOutputS3Uri",
            "Value": V3_PIPELINE_OUTPUT_URI,
        },
    ],
)

V3_EXECUTION_ARN = v3_start["PipelineExecutionArn"]
print("Started v3 failure demonstration:", V3_EXECUTION_ARN)

Started v3 failure demonstration: arn:aws:sagemaker:us-east-1:182280021178:pipeline/rf-intrusion-cnn-artifact-promotion-batch-transform/execution/i8b3lxk98azt


In [36]:
EXECUTION_ARN = V3_EXECUTION_ARN

execution_details = sm.describe_pipeline_execution(
    PipelineExecutionArn=EXECUTION_ARN
)

print("Execution ARN:", EXECUTION_ARN)
print("Status:", execution_details["PipelineExecutionStatus"])
print("Started:", execution_details["CreationTime"])
print("Failure reason:", execution_details.get("FailureReason", "None"))

step_response = sm.list_pipeline_execution_steps(
    PipelineExecutionArn=EXECUTION_ARN
)

print("\nPipeline steps:")
for step in step_response["PipelineExecutionSteps"]:
    print(
        f"- {step['StepName']}: {step['StepStatus']}"
        f" | {step.get('FailureReason', '')}"
    )

Execution ARN: arn:aws:sagemaker:us-east-1:182280021178:pipeline/rf-intrusion-cnn-artifact-promotion-batch-transform/execution/i8b3lxk98azt
Status: Failed
Started: 2026-10-12 20:50:02.218000+00:00
Failure reason: Step failure: One or multiple steps failed.

Pipeline steps:
- RFIntrusionPerformanceGateFailed: Failed | RF Intrusion promotion failed. ModelVersion= v3 accuracy= 0.6204166666666666 required_accuracy>= 0.87 macro_f1= 0.593101346243364 required_macro_f1>= 0.86
- RFIntrusionPerformanceGate: Succeeded | 
- EvaluateAndPackageRFIntrusionCNN: Succeeded | 
- ValidateRFIntrusionInputs: Succeeded | 


In [37]:
from urllib.parse import urlparse

metrics_uri = f"{V3_PIPELINE_OUTPUT_URI}/evaluation/metrics.json"
parsed = urlparse(metrics_uri)

metrics = json.loads(
    s3_readonly.get_object(
        Bucket=parsed.netloc,
        Key=parsed.path.lstrip("/"),
    )["Body"].read()
)

print(json.dumps(
    metrics["multiclass_classification_metrics"],
    indent=2,
))

{
  "accuracy": {
    "value": 0.6204166666666666
  },
  "macro_f1": {
    "value": 0.593101346243364
  },
  "macro_precision": {
    "value": 0.6184069350789572
  },
  "macro_recall": {
    "value": 0.6204166666666666
  }
}
